# Treino e avaliação QLoRA no Colab

Execute as células nesta ordem.
O projeto precisa estar no diretório do Colab ou ser enviado como pasta compactada.
Ative uma GPU T4 em: Ambiente de execução > Alterar tipo de ambiente > T4 GPU.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Ative uma GPU T4 antes de continuar.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ajuste esta pasta se o projeto estiver em outro diretório do Drive.
%cd /content/drive/MyDrive/trabalho-llm

In [ ]:
!pip install -r requirements.txt
!python src/data/generate_dataset.py --seed 42 --output data/raw/dataset.jsonl
!python src/data/split_dataset.py --input data/raw/dataset.jsonl --output-dir data/processed --seed 42
!python src/data/validate_dataset.py

In [ ]:
# Compara r=8 e r=16. A execução final deixa o adaptador r=16 em outputs/adapter.
!python src/train/train_qlora.py --config configs/train.yaml --compare-hparams

In [ ]:
!python src/eval/run_inference.py --test data/processed/test.jsonl --adapter outputs/adapter
!python src/eval/compare.py

In [ ]:
import json
from pathlib import Path

results = json.loads(Path('outputs/eval/results.json').read_text(encoding='utf-8'))
print(json.dumps(results, ensure_ascii=False, indent=2))

In [ ]:
!zip -r fine_tuned_outputs.zip outputs/adapter outputs/curves outputs/eval REPORT.md
from google.colab import files
files.download('fine_tuned_outputs.zip')